# 04 — Model Evaluation & Prediction

**职责**: 加载模型 → 评估性能 → 生成可视化报告 → 在测试集上预测 → 输出提交文件

**输入**: `outputs/models/*.pkl`, `data/raw/test_data.csv`  
**输出**: `outputs/predictions/submission.csv`, `outputs/figures/evaluation_*.png`

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from config import DATA_PROC, DATA_TEST, TARGET, PRED_DIR, FIG_DIR
from src.preprocessing import load_and_clean
from src.models import load_model
from src.evaluate import cross_validate_all, results_to_dataframe

## 1. 加载数据 & 模型

In [ ]:
df = pd.read_parquet(DATA_PROC)
X = df.drop(columns=[TARGET])
y = df[TARGET]

model_names = ['lasso', 'elasticnet', 'random_forest']
models = {name: load_model(name) for name in model_names}
print(f"Loaded models: {list(models.keys())}")

## 2. 模型对比评估

In [ ]:
results = cross_validate_all(models, X, y)
results_to_dataframe(results)

## 3. 残差分析

对最优模型做残差图，检查预测偏差。

In [ ]:
# TODO: 选择最优模型名称
best_name = 'random_forest'
best_model = models[best_name]

y_pred = best_model.predict(X)
residuals = y - y_pred

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle(f"Residual Analysis — {best_name}", fontsize=14, fontweight='bold')

ax = axes[0]
ax.scatter(y_pred, residuals, alpha=0.1, s=3, color='steelblue')
ax.axhline(0, color='crimson', linestyle='--')
ax.set_xlabel('Predicted (°C)'); ax.set_ylabel('Residual (°C)')
ax.set_title('Residuals vs Predicted')

ax = axes[1]
ax.hist(residuals, bins=60, color='steelblue', edgecolor='white')
ax.axvline(0, color='crimson', linestyle='--')
ax.set_xlabel('Residual (°C)'); ax.set_ylabel('Count')
ax.set_title('Residual Distribution')

FIG_DIR.mkdir(parents=True, exist_ok=True)
plt.tight_layout()
plt.savefig(FIG_DIR / 'evaluation_residuals.png', bbox_inches='tight')
plt.show()

## 4. 测试集预测 & 提交

In [ ]:
# TODO: 确认测试集的列与训练集对齐
# df_test = load_and_clean(DATA_TEST)
# X_test = df_test[X.columns]  # 保证列顺序一致
#
# predictions = best_model.predict(X_test)
#
# PRED_DIR.mkdir(parents=True, exist_ok=True)
# submission = pd.DataFrame({'id': df_test.index, TARGET: predictions})
# submission.to_csv(PRED_DIR / 'submission.csv', index=False)
# print(f"Saved predictions to {PRED_DIR / 'submission.csv'}")
# submission.head()